# 00: Peta induk seluruh eksperimen AR-LRX

**Tujuan.** Notebook ini memetakan setiap eksperimen dan audit di repositori ke berkas
hasilnya, ke tabel dan gambar disertasi, serta ke artefak paper IJIES. Dengan peta tersebut,
setiap angka yang dipertanggungjawabkan kepada promotor dapat ditelusuri sampai ke berkas
sumbernya.

**Notebook ini tidak menjalankan eksperimen apa pun.** Yang dilakukan hanya membaca
`results/`, membaca berkas kode sebagai teks, dan menampilkan tabel. Tidak ada model yang
dilatih dan tidak ada berkas di `results/`, `data/` atau `raw/` yang disentuh. Modul
`src/experiments` dan `tools` juga tidak diimpor, agar TensorFlow tidak ikut termuat.

**Isi**

| Bagian | Isi |
|---|---|
| 1 | Lingkungan dan cara memakai |
| 2 | Tabel induk seluruh eksperimen dan audit |
| 3 | Cek keberadaan setiap berkas hasil yang dirujuk |
| 4 | Cek kesetaraan `exp05c` dengan berkas yang dibaca Bab V |
| 5 | Peta tabel dan gambar disertasi → berkas hasil |
| 6 | Peta artefak paper IJIES → berkas hasil |
| 7 | Status seluruh ramalan pra-registrasi P1–P9 |
| 8 | Catatan reproduksi yang harus disampaikan apa adanya |
| 9 | Sidik jari integritas `results/` |

## 1. Lingkungan

In [1]:
# Notebook pelapor: HANYA MEMBACA.
# Tidak melatih model, tidak mengimpor src/experiments atau tools (agar TensorFlow
# tidak ikut termuat), dan tidak pernah menulis ke results/, data/ atau raw/.
%matplotlib inline
from pathlib import Path
import ast, json, hashlib, datetime, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def cari_root(p: Path) -> Path:
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "src" / "experiments").is_dir():
            return c
    raise RuntimeError("root repositori tidak ditemukan dari " + str(p))


ROOT = cari_root(Path.cwd().resolve())
RES = ROOT / "results"
n_berkas = sum(1 for f in RES.rglob("*") if f.is_file())

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.max_rows", 200)

print("Root repositori :", ROOT)
print("Folder hasil    :", RES.relative_to(ROOT), f"({n_berkas} berkas)")
print("Dijalankan      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))


def idn(x, d=2, tanda=False):
    """Format angka gaya Indonesia (koma desimal, titik ribuan)."""
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    if tanda and x > 0:
        s = "+" + s
    return s.replace("-", "\u2212")


def baca(nama, **kw):
    """Baca satu berkas di results/ sebagai DataFrame. Read-only."""
    f = RES / nama
    if not f.exists():
        raise FileNotFoundError(f"{nama} tidak ada di results/")
    return pd.read_csv(f, **kw)


def potongan(relpath, nama=None, awal=None, akhir=None):
    """Tampilkan potongan kode dengan membaca berkas sebagai TEKS.

    Memakai ``ast`` untuk menemukan batas fungsi/konstanta, sehingga modul
    eksperimen tidak perlu diimpor.
    """
    f = ROOT / relpath
    teks = f.read_text(encoding="utf-8")
    baris = teks.splitlines()
    if nama is not None:
        pohon = ast.parse(teks)
        ketemu = False
        for n in ast.walk(pohon):
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.name == nama:
                awal, akhir, ketemu = n.lineno, n.end_lineno, True
                break
            if isinstance(n, ast.Assign) and any(getattr(t, "id", None) == nama for t in n.targets):
                awal, akhir, ketemu = n.lineno, n.end_lineno, True
                break
        if not ketemu:
            raise KeyError(f"'{nama}' tidak ditemukan di {relpath}")
    awal = awal or 1
    akhir = akhir or len(baris)
    lebar = len(str(akhir))
    print(f"--- {relpath}:{awal}-{akhir} " + "-" * max(0, 74 - len(relpath)))
    for i in range(awal, akhir + 1):
        print(f"{str(i).rjust(lebar)} | {baris[i - 1]}")


def git(*args):
    """Jalankan perintah git read-only di root repositori."""
    try:
        r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if r.returncode == 0 else ""
    except Exception:
        return ""

Root repositori : I:\Shared drives\Riset S3\Experiment S3 Code\antgravity_riset\AR-LRX_hybrid-forecasting
Folder hasil    : results (179 berkas)
Dijalankan      : 2026-09-29 19:29


In [2]:
# Notebook ini tidak pernah menjalankan ulang eksperimen.
RUN_EXPERIMENT = False

PERINTAH = [
    "python tools/exp07_rolling_origin.py --part pharma-origins | pharma-seeds | rossmann-origins | rossmann-seeds | summary",
    "python tools/exp08_learner_stability.py --part pharma | rossmann | summary",
    "python tools/exp10_controlled_perturbation.py --part pharma | rossmann | summary",
    "python tools/exp11_pharma_calendar_s1.py",
    "python tools/exp12_pharma_baselines_completion.py",
    "python tools/exp13_pharma_lstm_wide.py",
    "python tools/exp14_pharma_ets.py",
    "python tools/groupB_inference.py",
    "python tools/crossfit_audit_pharma.py",
    "python tools/crossfit_audit_rossmann.py",
    "python tools/leak_size_audit.py",
    "python tools/icoict_leak_audit.py",
    "python tools/icoict_paper_config.py",
    "python tools/rerun_lightgbm_baseline.py",
]

if RUN_EXPERIMENT:
    raise RuntimeError(
        "Notebook peta tidak menjalankan eksperimen. Jalankan perintah aslinya di terminal, "
        "atau pakai notebook pelapor masing-masing eksperimen."
    )
print("RUN_EXPERIMENT = False - tidak ada eksperimen yang dijalankan ulang.")
print()
print("Perintah asli yang menghasilkan seluruh berkas di results/:")
for c in PERINTAH:
    print("   ", c)
print()
print("Notebook lama (exp01-exp06b, figures_paper) dijalankan dari notebooks/,")
print("urutan dan runtimenya ada di REPRODUCE.md bagian 1.")

RUN_EXPERIMENT = False - tidak ada eksperimen yang dijalankan ulang.

Perintah asli yang menghasilkan seluruh berkas di results/:
    python tools/exp07_rolling_origin.py --part pharma-origins | pharma-seeds | rossmann-origins | rossmann-seeds | summary
    python tools/exp08_learner_stability.py --part pharma | rossmann | summary
    python tools/exp10_controlled_perturbation.py --part pharma | rossmann | summary
    python tools/exp11_pharma_calendar_s1.py
    python tools/exp12_pharma_baselines_completion.py
    python tools/exp13_pharma_lstm_wide.py
    python tools/exp14_pharma_ets.py
    python tools/groupB_inference.py
    python tools/crossfit_audit_pharma.py
    python tools/crossfit_audit_rossmann.py
    python tools/leak_size_audit.py
    python tools/icoict_leak_audit.py
    python tools/icoict_paper_config.py
    python tools/rerun_lightgbm_baseline.py

Notebook lama (exp01-exp06b, figures_paper) dijalankan dari notebooks/,
urutan dan runtimenya ada di REPRODUCE.md bagian 

## 2. Tabel induk

Satu baris disediakan untuk setiap eksperimen dan setiap audit. Kolom `Status` memakai lima
kosakata. `terbukti` berarti ramalan atau klaimnya didukung berkas hasil. `GAGAL` berarti
ramalan pra-registrasi tidak terpenuhi, dan status itu ditulis apa adanya. `dasar` berarti
eksperimen menyediakan angka untuk eksperimen lain tanpa memiliki ramalan sendiri. `audit`
berarti skrip memeriksa prosedur, tanpa menguji hipotesis. `alat` berarti skrip bantu yang
tidak menghasilkan angka.

In [3]:
# Tabel induk. Disusun dari: judul subbab dan pemanggilan read_csv pada sumber
# naskah (content_bab4.py / content_bab5.py), REPRODUCE.md bagian 2 untuk paper
# IJIES, serta PREREG/RESULTS_exp10_exp13_exp14.md untuk status ramalan.
PETA = [
 dict(exp="exp01", tujuan="PharmaSales harian di bawah kontrak eksperimen C1-C8; di sini ARIMA(5,1,0) masih multi-langkah",
      skrip="notebooks/exp01_pharma_daily_unified.ipynb",
      hasil="exp01_pharma_daily*.csv, .meta.json",
      disertasi="Tabel 4.2; Subbab 4.2.1, 4.2.3", ijies="-",
      status="dasar",
      catatan="TIDAK mereproduksi baris arsip bila dijalankan apa adanya: baris 'LR-XGB (residual)' kini cross-fitted secara bawaan. Perlu P.fp_lr_xgb_residual(..., cross_fit_folds=None). REPRODUCE.md bagian 8."),
 dict(exp="exp02", tujuan="PharmaSales mingguan di bawah kontrak yang sama",
      skrip="notebooks/exp02_pharma_weekly_unified.ipynb",
      hasil="exp02_pharma_weekly*.csv, .meta.json",
      disertasi="Subbab 4.2.2, 4.2.3 (dalam teks)", ijies="-",
      status="dasar",
      catatan="Sama dengan exp01: baris LR-XGB (residual) tidak reproduksi apa adanya."),
 dict(exp="exp03", tujuan="Mengapa kolom Customers tidak boleh dipakai serentak; mendefinisikan varian V1/V2/V3",
      skrip="notebooks/exp03_rossmann_leakage_ablation.ipynb",
      hasil="exp03_rossmann_leakage_ablation*.csv, .meta.json",
      disertasi="Tabel 4.3; Subbab 4.2.4", ijies="-",
      status="dasar",
      catatan="Sama dengan exp01, ditambah baris LightGBM: B.fp_lightgbm kini subsample_freq=1. Perlu subsample_freq=0 untuk angka arsip."),
 dict(exp="exp04", tujuan="Merakit tabel-tabel studi awal",
      skrip="notebooks/exp04_paper_tables.ipynb",
      hasil="results/paper_tables/",
      disertasi="Subbab 4.1", ijies="-",
      status="dasar",
      catatan="Mewarisi caveat exp01-exp03."),
 dict(exp="exp05a", tujuan="AR-LRX pada panel ritel: tiga tahap pertama x tiga set fitur",
      skrip="notebooks/exp05a_rossmann_arlrx.ipynb",
      hasil="exp05a_rossmann_arlrx*.csv, .meta.json",
      disertasi="Dasar Subbab 4.4", ijies="acuan untuk exp05b",
      status="dasar",
      catatan="Sudah dijalankan ulang dengan residual cross-fitted (R1-1)."),
 dict(exp="exp05b", tujuan="Cek reproduksi, DM dua skala, ablation penuh, kurva gerbang, segmen",
      skrip="notebooks/exp05b_rossmann_arlrx_audit.ipynb",
      hasil="exp05b_rossmann_arlrx_audit*.csv, *_predictions.npz",
      disertasi="Tabel 4.5, 5.1; Gambar 4.2", ijies="Tabel 1, 2; Gambar 2, 3, 5",
      status="terbukti",
      catatan="*_predictions.npz tidak dilacak git (*.npz diabaikan); harus berasal dari run asli."),
 dict(exp="exp05c", tujuan="AR-LRX pada 32 konfigurasi PharmaSales (8 kategori ATC x 2 tingkat agregasi x 2 set fitur)",
      skrip="notebooks/exp05c_pharma_arlrx.ipynb",
      hasil="exp05c_pharma_arlrx*.csv, *_predictions.npz",
      disertasi="Tabel 4.10, 5.1-5.4, 5.6; Gambar 5.1-5.3", ijies="Tabel 1, 8, 9, 10; Gambar 3, 4, 5",
      status="terbukti",
      catatan="Bab V membaca exp05c_crossfit_crossfit.csv (keluaran crossfit_audit_pharma.py), bukan exp05c_pharma_arlrx.csv. Kesetaraan keduanya diperiksa pada bagian 4 notebook ini."),
 dict(exp="exp05d", tujuan="Varian augmented dan gerbang per segmen pada Rossmann",
      skrip="notebooks/exp05d_rossmann_arlrx_dev.ipynb",
      hasil="exp05d_rossmann_arlrx_dev*.csv, *_predictions.npz",
      disertasi="Tabel 4.5, 4.8, 5.5", ijies="Tabel 2, 3, 6",
      status="terbukti", catatan="-"),
 dict(exp="exp06", tujuan="Pelatihan model pembanding tahap pertama (versi lemah)",
      skrip="notebooks/exp06_rossmann_baselines_unified.ipynb",
      hasil="exp06_rossmann_baselines_unified*.csv",
      disertasi="Tabel 4.7", ijies="Tabel 5",
      status="dasar",
      catatan="Sengaja dipertahankan sebagai pembanding 'sebelum diperbaiki'. Baris LightGBM arsip berjalan TANPA row bagging."),
 dict(exp="exp06b", tujuan="Model pembanding dilatih ulang dengan benar; inilah pass yang dilaporkan",
      skrip="notebooks/exp06b_rossmann_baselines_strong.ipynb",
      hasil="exp06b_rossmann_baselines_strong*.csv, *_predictions.npz",
      disertasi="Tabel 4.7, 4.8; Gambar 4.3", ijies="Tabel 3, 5",
      status="terbukti",
      catatan="Runtime asli 19,1 jam. Baris LightGBM sudah diganti oleh rerun_lightgbm_baseline.py."),
 dict(exp="exp07", tujuan="Replikasi pada empat forecast origin dan lima seed (R1-6, R2-1, R2-4)",
      skrip="tools/exp07_rolling_origin.py",
      hasil="exp07_*_origins.csv, exp07_*_seeds.csv, exp07_summary_*.csv",
      disertasi="Tabel 5.7, 5.8; Gambar 5.4", ijies="Tabel 7, 11",
      status="terbukti",
      catatan="PERHATIAN: angka Tabel 5.7 dan 5.8 ditulis sebagai LITERAL di content_bab5.py, tidak dibaca dari CSV. Kecocokannya harus diperiksa satu per satu di notebook exp07."),
 dict(exp="exp08", tujuan="Apakah label kondisi residual sifat residual atau sifat XGBoost? Enam jenis model (R2-2)",
      skrip="tools/exp08_learner_stability.py",
      hasil="exp08_learner_stability.csv, exp08_summary.csv",
      disertasi="Tabel 5.9; Subbab 5.6", ijies="Subbab 4.6",
      status="terbukti",
      catatan="Tabel 5.9 dibaca langsung dari exp08_summary.csv."),
 dict(exp="exp09", tujuan="(tidak ada) nomor tidak terpakai",
      skrip="-", hasil="-", disertasi="-", ijies="-",
      status="-",
      catatan="Penomoran melompat dari exp08 ke exp10. Dicatat agar tidak disangka ada eksperimen yang hilang."),
 dict(exp="exp10", tujuan="Uji terkontrol: noise bertingkat dan guncangan permintaan pada kedua dataset",
      skrip="tools/exp10_controlled_perturbation.py",
      hasil="exp10_pharma.csv, exp10_rossmann.csv, exp10_summary_*.csv",
      disertasi="Tabel 5.10, 5.11, 5.12; Gambar 5.5; Subbab 5.7", ijies="-",
      status="campuran: P6b, P7a-d, P8a-b terbukti; P6a dan P9b GAGAL",
      catatan="Pra-registrasi. Tidak ada parameter yang diubah setelah pra-registrasi dan tidak ada run yang diulang."),
 dict(exp="exp11", tujuan="Tahap pertama struktural berbasis kunci kalender pada PharmaSales",
      skrip="tools/exp11_pharma_calendar_s1.py",
      hasil="exp11_pharma_calendar*.csv, .meta.json",
      disertasi="Tabel 4.12; Subbab 4.6.3", ijies="-",
      status="terbukti",
      catatan="Ramalan tercatat di docstring skrip, bukan berkas pra-registrasi terpisah. Skrip dan hasil masuk pada commit yang sama (e4c061d), sehingga urutan waktu antara ramalan dan hasil tidak terdokumentasi."),
 dict(exp="exp12", tujuan="Melengkapi model pembanding PharmaSales: ARIMA bergulir satu langkah, Random Forest, LSTM",
      skrip="tools/exp12_pharma_baselines_completion.py",
      hasil="exp12_pharma_baselines*.csv, .meta.json",
      disertasi="Tabel 4.11; Subbab 4.6.2", ijies="-",
      status="terbukti",
      catatan="Membetulkan ketimpangan C7: ARIMA exp01 multi-langkah, model lain bergulir satu langkah."),
 dict(exp="exp13", tujuan="Apakah LSTM membaik bila grid lebarnya diperluas ke {32, ..., 512}?",
      skrip="tools/exp13_pharma_lstm_wide.py",
      hasil="exp13_pharma_lstm_wide*.csv, .meta.json",
      disertasi="Tabel 4.11; Subbab 4.6.2", ijies="-",
      status="terbukti (P4a, P4b)",
      catatan="Pra-registrasi. Memakai Keras/TensorFlow: tidak reproduksi bit-per-bit bahkan pada satu mesin."),
 dict(exp="exp14", tujuan="Pemulusan eksponensial ETS bergulir sebagai pembanding statistik yang paling mapan",
      skrip="tools/exp14_pharma_ets.py",
      hasil="exp14_pharma_ets*.csv, .meta.json",
      disertasi="Tabel 4.11, 4.13; Subbab 4.6.2, 4.7.1", ijies="-",
      status="GAGAL (P5a dan P5b, pada data harian)",
      catatan="Pra-registrasi. ETS lebih akurat daripada AR-LRX pada data obat harian; dilaporkan apa adanya."),
 dict(exp="crossfit_audit_pharma", tujuan="Audit sebelum/sesudah residual cross-fitted pada exp05c (R1-1)",
      skrip="tools/crossfit_audit_pharma.py",
      hasil="exp05c_crossfit_legacy*.csv, exp05c_crossfit_crossfit*.csv, exp05c_crossfit_audit_summary.csv",
      disertasi="Subbab 4.4.4 (teks); sumber angka Bab V", ijies="Subbab 2.3 (R1-1)",
      status="audit",
      catatan="Mode legacy (cross_fit_folds=None) harus mereproduksi arsip bit-per-bit; skrip memeriksanya sendiri."),
 dict(exp="crossfit_audit_rossmann", tujuan="Audit yang sama pada panel Rossmann",
      skrip="tools/crossfit_audit_rossmann.py",
      hasil="rossmann_crossfit_audit.csv",
      disertasi="Subbab 4.4.4 (teks)", ijies="Subbab 2.3 (R1-1)",
      status="audit",
      catatan="Kedua sisi harus dijalankan pada satu lingkungan; beda Windows/Linux mengubah digit signifikan keempat."),
 dict(exp="leak_size_audit", tujuan="Seberapa besar kebocoran residual in-sample tahap pertama",
      skrip="tools/leak_size_audit.py",
      hasil="crossfit_leak_size.csv",
      disertasi="Tabel 4.6; Subbab 4.4.4", ijies="Subbab 2.3",
      status="audit",
      catatan="Tidak melatih tahap kedua sama sekali."),
 dict(exp="icoict_leak_audit", tujuan="Audit kebocoran pipeline studi awal ICoICT 2025 (rolling mean memuat y_t)",
      skrip="tools/icoict_leak_audit.py",
      hasil="icoict_leak_audit.csv",
      disertasi="Tabel 4.1; Gambar 4.1; Subbab 4.1", ijies="-",
      status="audit",
      catatan="Varian A = notebook apa adanya (bocor); varian B = rolling mean digeser satu langkah."),
 dict(exp="icoict_paper_config", tujuan="Konfigurasi sesuai yang dinyatakan Bagian II paper ICoICT",
      skrip="tools/icoict_paper_config.py",
      hasil="icoict_paper_config.csv",
      disertasi="Tabel 4.1; Gambar 4.1; Subbab 4.1.3", ijies="-",
      status="audit",
      catatan="Sumber langsung angka Tabel 4.1 (variabel 'ic' di content_bab4.py)."),
 dict(exp="groupB_inference", tujuan="Inferensi yang sadar dependensi dan multiplisitas (R1-4, R2-5)",
      skrip="tools/groupB_inference.py",
      hasil="groupB_rossmann_dependence.csv, groupB_pharma_*.csv, groupB_summary.json",
      disertasi="Tabel 4.9; Gambar 4.4; Subbab 4.5.3", ijies="Tabel 4, 10",
      status="terbukti",
      catatan="Tidak melatih apa pun; hanya membaca *_predictions.npz yang tidak dilacak git."),
 dict(exp="rerun_lightgbm_baseline", tujuan="Melatih ulang baseline LightGBM dengan row bagging yang benar-benar aktif",
      skrip="tools/rerun_lightgbm_baseline.py",
      hasil="lightgbm_fix_audit.csv, *.before_lgbm_fix",
      disertasi="Tabel 4.7", ijies="Tabel 5",
      status="audit",
      catatan="subsample=0.8 tanpa subsample_freq diabaikan LightGBM. Skrip memeriksa reproduksi setelan lama lebih dahulu, lalu mengganti HANYA baris LightGBM."),
 dict(exp="fix_determinism", tujuan="Audit dan perbaikan determinisme notebook (random_state, seed global, salah skala pada predict)",
      skrip="tools/fix_determinism.py, tools/determinism_fixes.patch",
      hasil="(tidak ada berkas di results/)",
      disertasi="Kontrak C4", ijies="Subbab 3 (kontrak)",
      status="alat",
      catatan="Alat bantu, bukan eksperimen. Tidak menghasilkan angka."),
]

peta = pd.DataFrame(PETA)
peta.columns = ["Eksperimen", "Tujuan", "Skrip / notebook", "Berkas hasil",
                "Disertasi (tabel / gambar)", "Paper IJIES", "Status", "Catatan reproduksi"]
print(f"{len(peta)} baris pada tabel induk.\n")
display(peta[["Eksperimen", "Tujuan", "Skrip / notebook", "Status"]])

26 baris pada tabel induk.



,Eksperimen,Tujuan,Skrip / notebook,Status
0,exp01,"PharmaSales harian di bawah kontrak eksperimen C1-C8; di sini ARIMA(5,1,0) masih multi-langkah",notebooks/exp01_pharma_daily_unified.ipynb,dasar
1,exp02,PharmaSales mingguan di bawah kontrak yang sama,notebooks/exp02_pharma_weekly_unified.ipynb,dasar
2,exp03,Mengapa kolom Customers tidak boleh dipakai serentak; mendefinisikan varian V1/V2/V3,notebooks/exp03_rossmann_leakage_ablation.ipynb,dasar
3,exp04,Merakit tabel-tabel studi awal,notebooks/exp04_paper_tables.ipynb,dasar
4,exp05a,AR-LRX pada panel ritel: tiga tahap pertama x tiga set fitur,notebooks/exp05a_rossmann_arlrx.ipynb,dasar
5,exp05b,"Cek reproduksi, DM dua skala, ablation penuh, kurva gerbang, segmen",notebooks/exp05b_rossmann_arlrx_audit.ipynb,terbukti
6,exp05c,AR-LRX pada 32 konfigurasi PharmaSales (8 kategori ATC x 2 tingkat agregasi x 2 set fitur),notebooks/exp05c_pharma_arlrx.ipynb,terbukti
7,exp05d,Varian augmented dan gerbang per segmen pada Rossmann,notebooks/exp05d_rossmann_arlrx_dev.ipynb,terbukti
8,exp06,Pelatihan model pembanding tahap pertama (versi lemah),notebooks/exp06_rossmann_baselines_unified.ipynb,dasar
9,exp06b,Model pembanding dilatih ulang dengan benar; inilah pass yang dilaporkan,notebooks/exp06b_rossmann_baselines_strong.ipynb,terbukti


Kolom yang panjang ditampilkan per eksperimen agar terbaca utuh.

In [4]:
# Kolom yang panjang ditampilkan terpisah agar terbaca.
for _, r in peta.iterrows():
    print("=" * 100)
    print(f"{r['Eksperimen']}  [{r['Status']}]")
    print(f"  Skrip      : {r['Skrip / notebook']}")
    print(f"  Hasil      : {r['Berkas hasil']}")
    print(f"  Disertasi  : {r['Disertasi (tabel / gambar)']}")
    print(f"  IJIES      : {r['Paper IJIES']}")
    print(f"  Reproduksi : {r['Catatan reproduksi']}")

exp01  [dasar]
  Skrip      : notebooks/exp01_pharma_daily_unified.ipynb
  Hasil      : exp01_pharma_daily*.csv, .meta.json
  Disertasi  : Tabel 4.2; Subbab 4.2.1, 4.2.3
  IJIES      : -
  Reproduksi : TIDAK mereproduksi baris arsip bila dijalankan apa adanya: baris 'LR-XGB (residual)' kini cross-fitted secara bawaan. Perlu P.fp_lr_xgb_residual(..., cross_fit_folds=None). REPRODUCE.md bagian 8.
exp02  [dasar]
  Skrip      : notebooks/exp02_pharma_weekly_unified.ipynb
  Hasil      : exp02_pharma_weekly*.csv, .meta.json
  Disertasi  : Subbab 4.2.2, 4.2.3 (dalam teks)
  IJIES      : -
  Reproduksi : Sama dengan exp01: baris LR-XGB (residual) tidak reproduksi apa adanya.
exp03  [dasar]
  Skrip      : notebooks/exp03_rossmann_leakage_ablation.ipynb
  Hasil      : exp03_rossmann_leakage_ablation*.csv, .meta.json
  Disertasi  : Tabel 4.3; Subbab 4.2.4
  IJIES      : -
  Reproduksi : Sama dengan exp01, ditambah baris LightGBM: B.fp_lightgbm kini subsample_freq=1. Perlu subsample_freq=0 untuk a

## 3. Cek keberadaan berkas hasil

Setiap berkas yang dirujuk tabel induk diperiksa. Berkas yang tidak ada harus punya
alasan yang tercatat (misalnya `*.npz` dan `*.log` yang diabaikan git).

In [5]:
# Setiap pola berkas pada tabel induk diperiksa keberadaannya di results/.
POLA = {
 "exp01": ["exp01_pharma_daily.csv", "exp01_pharma_daily.meta.json", "exp01_pharma_daily_dm_test.csv",
           "exp01_pharma_daily_lag_ablation.csv", "exp01_pharma_daily_splits.csv"],
 "exp02": ["exp02_pharma_weekly.csv", "exp02_pharma_weekly.meta.json", "exp02_pharma_weekly_dm_test.csv",
           "exp02_pharma_weekly_lag_ablation.csv", "exp02_pharma_weekly_splits.csv"],
 "exp03": ["exp03_rossmann_leakage_ablation.csv", "exp03_rossmann_leakage_ablation.meta.json",
           "exp03_rossmann_leakage_ablation_dm_test.csv", "exp03_rossmann_leakage_ablation_splits.csv"],
 "exp04": ["paper_tables"],
 "exp05a": ["exp05a_rossmann_arlrx.csv", "exp05a_rossmann_arlrx.meta.json",
            "exp05a_rossmann_arlrx_ablation.csv", "exp05a_rossmann_arlrx_dm_test.csv",
            "exp05a_rossmann_arlrx_summary.csv"],
 "exp05b": ["exp05b_rossmann_arlrx_audit.csv", "exp05b_rossmann_arlrx_audit.meta.json",
            "exp05b_rossmann_arlrx_audit_ablation_full.csv", "exp05b_rossmann_arlrx_audit_dm_two_scales.csv",
            "exp05b_rossmann_arlrx_audit_gate_curve.csv", "exp05b_rossmann_arlrx_audit_segments.csv",
            "exp05b_rossmann_arlrx_audit_predictions.npz"],
 "exp05c": ["exp05c_pharma_arlrx.csv", "exp05c_pharma_arlrx.meta.json", "exp05c_pharma_arlrx_comparison.csv",
            "exp05c_pharma_arlrx_dm.csv", "exp05c_pharma_arlrx_gate.csv", "exp05c_pharma_arlrx_aug_effect.csv",
            "exp05c_pharma_arlrx_predictions.npz"],
 "exp05d": ["exp05d_rossmann_arlrx_dev.csv", "exp05d_rossmann_arlrx_dev.meta.json",
            "exp05d_rossmann_arlrx_dev_main.csv", "exp05d_rossmann_arlrx_dev_dm.csv",
            "exp05d_rossmann_arlrx_dev_ablation_2x2.csv", "exp05d_rossmann_arlrx_dev_segments.csv",
            "exp05d_rossmann_arlrx_dev_predictions.npz"],
 "exp06": ["exp06_rossmann_baselines_unified.csv", "exp06_rossmann_baselines_unified.meta.json",
           "exp06_rossmann_baselines_unified_dm.csv", "exp06_rossmann_baselines_unified_table_utama.csv",
           "exp06_rossmann_baselines_unified_lama_vs_adil.csv"],
 "exp06b": ["exp06b_rossmann_baselines_strong.csv", "exp06b_rossmann_baselines_strong.meta.json",
            "exp06b_rossmann_baselines_strong_dm.csv", "exp06b_rossmann_baselines_strong_table_utama.csv",
            "exp06b_rossmann_baselines_strong_exp06_vs_exp06b.csv"],
 "exp07": ["exp07_pharma_origins.csv", "exp07_pharma_seeds.csv", "exp07_rossmann_origins.csv",
           "exp07_rossmann_seeds.csv", "exp07_summary_pharma_origins.csv", "exp07_summary_pharma_seeds.csv",
           "exp07_summary_rossmann_origins.csv", "exp07_summary_rossmann_seeds.csv"],
 "exp08": ["exp08_learner_stability.csv", "exp08_summary.csv"],
 "exp10": ["exp10_pharma.csv", "exp10_rossmann.csv", "exp10_summary_pharma.csv", "exp10_summary_rossmann.csv"],
 "exp11": ["exp11_pharma_calendar.csv", "exp11_pharma_calendar.meta.json", "exp11_pharma_calendar_dm.csv",
           "exp11_pharma_calendar_summary.csv"],
 "exp12": ["exp12_pharma_baselines.csv", "exp12_pharma_baselines.meta.json",
           "exp12_pharma_baselines_dm.csv", "exp12_pharma_baselines_summary.csv"],
 "exp13": ["exp13_pharma_lstm_wide.csv", "exp13_pharma_lstm_wide.meta.json", "exp13_pharma_lstm_wide_dm.csv",
           "exp13_pharma_lstm_wide_summary.csv", "exp13_pharma_lstm_wide_vs_exp12.csv"],
 "exp14": ["exp14_pharma_ets.csv", "exp14_pharma_ets.meta.json", "exp14_pharma_ets_dm.csv",
           "exp14_pharma_ets_summary.csv"],
 "crossfit_audit_pharma": ["exp05c_crossfit_legacy.csv", "exp05c_crossfit_crossfit.csv",
                           "exp05c_crossfit_audit_summary.csv"],
 "crossfit_audit_rossmann": ["rossmann_crossfit_audit.csv"],
 "leak_size_audit": ["crossfit_leak_size.csv"],
 "icoict_leak_audit": ["icoict_leak_audit.csv"],
 "icoict_paper_config": ["icoict_paper_config.csv"],
 "groupB_inference": ["groupB_rossmann_dependence.csv", "groupB_pharma_tests_bh.csv",
                      "groupB_pharma_config_ci.csv", "groupB_pharma_pooled.csv", "groupB_summary.json"],
 "rerun_lightgbm_baseline": ["lightgbm_fix_audit.csv"],
}

baris = []
for exp, berkas in POLA.items():
    for b in berkas:
        p = RES / b
        baris.append({"Eksperimen": exp, "Berkas": b,
                      "Ada": "ya" if p.exists() else "TIDAK",
                      "Ukuran (KB)": round(p.stat().st_size / 1024, 1) if p.is_file() else
                                     (f"{sum(1 for _ in p.rglob('*') if _.is_file())} berkas" if p.is_dir() else "-")})
cekb = pd.DataFrame(baris)
hilang = cekb[cekb.Ada == "TIDAK"]
print(f"{len(cekb)} berkas dirujuk, {len(cekb) - len(hilang)} ada, {len(hilang)} tidak ada.")
print()
if len(hilang):
    print("Berkas yang TIDAK ada (diperiksa apakah memang diabaikan git):")
    display(hilang)
else:
    print("Seluruh berkas yang dirujuk tabel induk ada di results/.")
display(cekb)

95 berkas dirujuk, 95 ada, 0 tidak ada.

Seluruh berkas yang dirujuk tabel induk ada di results/.


,Eksperimen,Berkas,Ada,Ukuran (KB)
0,exp01,exp01_pharma_daily.csv,ya,66.9
1,exp01,exp01_pharma_daily.meta.json,ya,0.5
2,exp01,exp01_pharma_daily_dm_test.csv,ya,1.9
3,exp01,exp01_pharma_daily_lag_ablation.csv,ya,36.1
4,exp01,exp01_pharma_daily_splits.csv,ya,2.0
5,exp02,exp02_pharma_weekly.csv,ya,66.6
6,exp02,exp02_pharma_weekly.meta.json,ya,0.5
7,exp02,exp02_pharma_weekly_dm_test.csv,ya,2.0
8,exp02,exp02_pharma_weekly_lag_ablation.csv,ya,35.8
9,exp02,exp02_pharma_weekly_splits.csv,ya,1.9


## 4. Cek kesetaraan exp05c

Bab V disertasi membaca `exp05c_crossfit_crossfit.csv`, yaitu keluaran
`crossfit_audit_pharma.py`, dan tidak membaca `exp05c_pharma_arlrx.csv` yang dihasilkan
notebook exp05c. Sejak revisi R1-1 keduanya menjalankan prosedur yang sama, sehingga baris
yang sama harus memberi `test_RMSE` yang sama. Kesetaraan itu diperiksa pada sel berikut.

In [6]:
# Bab V membaca exp05c_crossfit_crossfit.csv, bukan exp05c_pharma_arlrx.csv.
# Keduanya seharusnya menjalankan prosedur yang sama (residual cross-fitted),
# jadi baris yang sama harus memberi test_RMSE yang sama.
kunci = ["granularity", "category", "feature_set", "model"]
a = baca("exp05c_pharma_arlrx.csv")[kunci + ["test_RMSE"]]
b = baca("exp05c_crossfit_crossfit.csv")[kunci + ["test_RMSE"]]
m = a.merge(b, on=kunci, suffixes=("_exp05c", "_crossfit"))
m["|selisih|"] = (m.test_RMSE_exp05c - m.test_RMSE_crossfit).abs()

print(f"Model yang ada di kedua berkas : {sorted(m.model.unique())}")
print(f"Baris yang berpasangan        : {len(m)}")
print(f"Selisih |test_RMSE| terbesar  : {m['|selisih|'].max():.3e}")
print()
if m["|selisih|"].max() < 1e-9:
    print("IDENTIK. Angka Bab V dan angka exp05c berasal dari prosedur yang sama.")
else:
    print("ADA SELISIH. Perlu dilaporkan, jangan dibetulkan diam-diam:")
    display(m.nlargest(10, "|selisih|"))

Model yang ada di kedua berkas : ['AR-LRX [linear]', 'AR-LRX-Aug [linear]', 'AR-LRX-g [linear]', 'LR-XGB (residual, tanpa gerbang)', 'Naive', 'S1 [linear]', 'SeasonalNaive(s=52)', 'SeasonalNaive(s=7)', 'XGBoost']
Baris yang berpasangan        : 256
Selisih |test_RMSE| terbesar  : 0.000e+00

IDENTIK. Angka Bab V dan angka exp05c berasal dari prosedur yang sama.


## 5. Peta tabel dan gambar disertasi

In [7]:
# Peta tabel dan gambar disertasi -> berkas hasil.
# Disusun dari pemanggilan read_csv dan pemakaian variabelnya di
# content_bab4.py dan content_bab5.py.
BAB4 = [
 ("Tabel 4.1",  "Reproduksi studi awal: fitur bocor vs fitur diperbaiki", "icoict_paper_config.csv"),
 ("Gambar 4.1", "RMSE hybrid studi awal: dilaporkan / bocor / diperbaiki", "icoict_paper_config.csv"),
 ("Tabel 4.2",  "RMSE uji PharmaSales harian di bawah kontrak, set fitur A", "exp01_pharma_daily.csv"),
 ("Tabel 4.3",  "RMSE uji Rossmann skala asli pada empat varian fitur", "exp03_rossmann_leakage_ablation.csv"),
 ("Tabel 4.4",  "Diagnosis kegagalan kerangka lama (naratif)", "- (tidak ada berkas)"),
 ("Tabel 4.5",  "Ablation study komponen AR-LRX pada Rossmann V3", "exp05b_..._ablation_full.csv, exp05d_rossmann_arlrx_dev.csv"),
 ("Gambar 4.2", "Kurva RMSE terhadap bobot gerbang w", "exp05b_rossmann_arlrx_audit_gate_curve.csv"),
 ("Tabel 4.6",  "Besar kebocoran residual tahap pertama", "crossfit_leak_size.csv"),
 ("Tabel 4.7",  "Dampak penguatan pelatihan model pembanding", "exp06b_..._exp06_vs_exp06b.csv, lightgbm_fix_audit.csv"),
 ("Tabel 4.8",  "Seluruh model di bawah protokol terpadu, Rossmann V3", "exp06b_..._table_utama.csv"),
 ("Gambar 4.3", "RMSE uji seluruh model pada Rossmann V3", "exp06b_..._table_utama.csv"),
 ("Tabel 4.9",  "Inferensi AR-LRX terhadap setiap model", "groupB_rossmann_dependence.csv"),
 ("Gambar 4.4", "Selisih RMSE dengan selang kepercayaan bootstrap", "groupB_rossmann_dependence.csv"),
 ("Tabel 4.10", "Akurasi AR-LRX pada 32 konfigurasi PharmaSales", "exp05c_pharma_arlrx_comparison.csv, _dm.csv"),
 ("Gambar 4.5", "Contoh prediksi satu langkah, kategori R06 mingguan", "exp05c_pharma_arlrx_predictions.npz"),
 ("Tabel 4.11", "AR-LRX terhadap pembanding tambahan (exp12-exp14)",
                "exp12_pharma_baselines_summary.csv, exp13_pharma_lstm_wide_summary.csv, exp14_pharma_ets_summary.csv, exp13_..._vs_exp12.csv"),
 ("Tabel 4.12", "Model dengan fitur kalender (exp11)", "exp11_pharma_calendar_summary.csv"),
 ("Tabel 4.13", "Ringkasan penurunan error AR-LRX (RQ1)", "gabungan exp05c, exp06b, exp12-exp14"),
 ("Tabel 4.14", "Perbandingan rancangan dengan hybrid terdahulu (naratif)", "- (tidak ada berkas)"),
]
BAB5 = [
 ("Tabel 5.1",  "R2 tahap kedua pada residual validasi di kedua dataset", "exp05b_..._audit.csv, exp05c_crossfit_crossfit.csv"),
 ("Gambar 5.1", "Sebaran R2 residual validasi", "exp05b_..._audit.csv, exp05c_crossfit_crossfit.csv"),
 ("Tabel 5.2",  "Hasil pengujian tiga ramalan P1-P3 pada PharmaSales", "exp05c_crossfit_crossfit.csv"),
 ("Tabel 5.3",  "Sebaran bobot gerbang terpilih", "exp05c_crossfit_crossfit.csv"),
 ("Tabel 5.4",  "Pemburukan terhadap tahap pertama linier", "exp05c_crossfit_crossfit.csv"),
 ("Gambar 5.2", "Perubahan RMSE uji terhadap tahap pertama linier", "exp05c_crossfit_crossfit.csv"),
 ("Gambar 5.3", "Perbaikan gerbang pada validasi vs pada uji", "exp05c_crossfit_crossfit.csv"),
 ("Tabel 5.5",  "RMSE uji per segmen, Rossmann V3", "exp05d_rossmann_arlrx_dev_segments.csv"),
 ("Tabel 5.6",  "Perilaku gerbang per kategori ATC dan tingkat agregasi", "exp05c_crossfit_crossfit.csv"),
 ("Tabel 5.7",  "Replikasi pada Rossmann V3 (origin dan seed)", "exp07_summary_rossmann_origins.csv, _seeds.csv  [LITERAL di naskah]"),
 ("Gambar 5.4", "RMSE uji pada empat origin dan lima seed", "exp07_rossmann_origins.csv, _seeds.csv"),
 ("Tabel 5.8",  "Stabilitas label kondisi residual dan gerbang, PharmaSales", "exp07_summary_pharma_origins.csv, _seeds.csv  [LITERAL di naskah]"),
 ("Tabel 5.9",  "R2 model residual untuk enam jenis model", "exp08_summary.csv"),
 ("Tabel 5.10", "AR-LRX bergerbang di bawah gangguan terkontrol, PharmaSales", "exp10_summary_pharma.csv"),
 ("Tabel 5.11", "AR-LRX augmented di bawah gangguan terkontrol, Rossmann", "exp10_summary_rossmann.csv"),
 ("Gambar 5.5", "Respons Rossmann terhadap noise terkontrol", "exp10_summary_rossmann.csv"),
 ("Tabel 5.12", "Hasil ramalan pra-registrasi P4-P9", "exp10_summary_*.csv, exp13_..._summary.csv, exp14_..._summary.csv"),
 ("Tabel 5.13", "Ringkasan ketahanan sepanjang tujuh aspek uji (RQ2)", "gabungan exp05c-exp10"),
]
t4 = pd.DataFrame(BAB4, columns=["Artefak", "Isi", "Berkas hasil"])
t5 = pd.DataFrame(BAB5, columns=["Artefak", "Isi", "Berkas hasil"])
print("BAB IV (RQ1)"); display(t4)
print("BAB V (RQ2)"); display(t5)
print()
print("Catatan: '[LITERAL di naskah]' berarti angka tabel itu diketik langsung di")
print("content_bab5.py, bukan dibaca dari CSV, sehingga kecocokannya tidak otomatis.")

BAB IV (RQ1)


,Artefak,Isi,Berkas hasil
0,Tabel 4.1,Reproduksi studi awal: fitur bocor vs fitur diperbaiki,icoict_paper_config.csv
1,Gambar 4.1,RMSE hybrid studi awal: dilaporkan / bocor / diperbaiki,icoict_paper_config.csv
2,Tabel 4.2,"RMSE uji PharmaSales harian di bawah kontrak, set fitur A",exp01_pharma_daily.csv
3,Tabel 4.3,RMSE uji Rossmann skala asli pada empat varian fitur,exp03_rossmann_leakage_ablation.csv
4,Tabel 4.4,Diagnosis kegagalan kerangka lama (naratif),- (tidak ada berkas)
5,Tabel 4.5,Ablation study komponen AR-LRX pada Rossmann V3,"exp05b_..._ablation_full.csv, exp05d_rossmann_arlrx_dev.csv"
6,Gambar 4.2,Kurva RMSE terhadap bobot gerbang w,exp05b_rossmann_arlrx_audit_gate_curve.csv
7,Tabel 4.6,Besar kebocoran residual tahap pertama,crossfit_leak_size.csv
8,Tabel 4.7,Dampak penguatan pelatihan model pembanding,"exp06b_..._exp06_vs_exp06b.csv, lightgbm_fix_audit.csv"
9,Tabel 4.8,"Seluruh model di bawah protokol terpadu, Rossmann V3",exp06b_..._table_utama.csv


BAB V (RQ2)


,Artefak,Isi,Berkas hasil
0,Tabel 5.1,R2 tahap kedua pada residual validasi di kedua dataset,"exp05b_..._audit.csv, exp05c_crossfit_crossfit.csv"
1,Gambar 5.1,Sebaran R2 residual validasi,"exp05b_..._audit.csv, exp05c_crossfit_crossfit.csv"
2,Tabel 5.2,Hasil pengujian tiga ramalan P1-P3 pada PharmaSales,exp05c_crossfit_crossfit.csv
3,Tabel 5.3,Sebaran bobot gerbang terpilih,exp05c_crossfit_crossfit.csv
4,Tabel 5.4,Pemburukan terhadap tahap pertama linier,exp05c_crossfit_crossfit.csv
5,Gambar 5.2,Perubahan RMSE uji terhadap tahap pertama linier,exp05c_crossfit_crossfit.csv
6,Gambar 5.3,Perbaikan gerbang pada validasi vs pada uji,exp05c_crossfit_crossfit.csv
7,Tabel 5.5,"RMSE uji per segmen, Rossmann V3",exp05d_rossmann_arlrx_dev_segments.csv
8,Tabel 5.6,Perilaku gerbang per kategori ATC dan tingkat agregasi,exp05c_crossfit_crossfit.csv
9,Tabel 5.7,Replikasi pada Rossmann V3 (origin dan seed),"exp07_summary_rossmann_origins.csv, _seeds.csv [LITERAL di naskah]"



Catatan: '[LITERAL di naskah]' berarti angka tabel itu diketik langsung di
content_bab5.py, bukan dibaca dari CSV, sehingga kecocokannya tidak otomatis.


## 6. Peta artefak paper IJIES

In [8]:
# Peta artefak paper IJIES -> berkas hasil, disalin dari REPRODUCE.md bagian 2.
IJIES = [
 ("Tabel 1",  "Dataset dan pembagian data", "exp05b, exp05c", "*.meta.json"),
 ("Tabel 2",  "Ablation komponen", "exp05b, exp05d", "exp05b_..._ablation_full.csv, exp05d_rossmann_arlrx_dev.csv"),
 ("Tabel 3",  "Seluruh model, protokol terpadu", "exp06b, exp05d", "exp06b_..._table_utama.csv"),
 ("Tabel 4",  "Inferensi yang sadar dependensi", "tools/groupB_inference.py", "groupB_rossmann_dependence.csv"),
 ("Tabel 5",  "Dampak pembetulan pelatihan pembanding", "exp06 -> exp06b", "exp06b_..._exp06_vs_exp06b.csv, lightgbm_fix_audit.csv"),
 ("Tabel 6",  "RMSE uji per segmen", "exp05d", "exp05d_..._segments.csv"),
 ("Tabel 7",  "Origin dan seed pada ritel", "tools/exp07_rolling_origin.py", "exp07_summary_rossmann_origins.csv, _seeds.csv"),
 ("Tabel 8",  "Bobot gerbang terpilih", "exp05c", "exp05c_pharma_arlrx_gate.csv"),
 ("Tabel 9",  "Pemburukan terhadap tahap pertama", "exp05c", "exp05c_..._comparison.csv, _gate.csv"),
 ("Tabel 10", "Akurasi terhadap empat pembanding", "exp05c, groupB_inference", "exp05c_..._comparison.csv, _dm.csv, groupB_pharma_*.csv"),
 ("Tabel 11", "Origin dan seed pada data obat", "tools/exp07_rolling_origin.py", "exp07_summary_pharma_origins.csv, _seeds.csv"),
 ("Subbab 4.6", "Stabilitas pemodel residual", "tools/exp08_learner_stability.py", "exp08_learner_stability.csv, exp08_summary.csv"),
 ("Subbab 2.3", "Besar kebocoran residual in-sample", "tools/leak_size_audit.py", "crossfit_leak_size.csv"),
 ("Gambar 1",  "Diagram kerangka", "digambar manual", "paper/figures/fig1_framework.png"),
 ("Gambar 2",  "Kurva sensitivitas gerbang", "figures_paper", "exp05b_..._gate_curve.csv"),
 ("Gambar 3",  "Diagnostik kondisi residual", "figures_paper", "exp05b_..._audit.csv, exp05d_..._dev.csv, exp05c_..._gate.csv"),
 ("Gambar 4",  "Sebaran pemburukan", "figures_paper", "exp05c_..._comparison.csv, _gate.csv"),
 ("Gambar 5",  "Indikator validasi lintas kondisi", "figures_paper", "exp05b_..._audit.csv, exp05c_..._gate.csv"),
 ("Subbab 5.3", "Perbandingan dengan nilai publikasi", "(tanpa run)", "published_baseline_values.csv"),
]
display(pd.DataFrame(IJIES, columns=["Artefak paper", "Isi", "Notebook / skrip", "Berkas hasil"]))
print()
print("exp10 sampai exp14 TIDAK dipakai paper IJIES: ketiganya dijalankan setelah")
print("naskah ronde 2 dikirim, dan hanya masuk disertasi.")

,Artefak paper,Isi,Notebook / skrip,Berkas hasil
0,Tabel 1,Dataset dan pembagian data,"exp05b, exp05c",*.meta.json
1,Tabel 2,Ablation komponen,"exp05b, exp05d","exp05b_..._ablation_full.csv, exp05d_rossmann_arlrx_dev.csv"
2,Tabel 3,"Seluruh model, protokol terpadu","exp06b, exp05d",exp06b_..._table_utama.csv
3,Tabel 4,Inferensi yang sadar dependensi,tools/groupB_inference.py,groupB_rossmann_dependence.csv
4,Tabel 5,Dampak pembetulan pelatihan pembanding,exp06 -> exp06b,"exp06b_..._exp06_vs_exp06b.csv, lightgbm_fix_audit.csv"
5,Tabel 6,RMSE uji per segmen,exp05d,exp05d_..._segments.csv
6,Tabel 7,Origin dan seed pada ritel,tools/exp07_rolling_origin.py,"exp07_summary_rossmann_origins.csv, _seeds.csv"
7,Tabel 8,Bobot gerbang terpilih,exp05c,exp05c_pharma_arlrx_gate.csv
8,Tabel 9,Pemburukan terhadap tahap pertama,exp05c,"exp05c_..._comparison.csv, _gate.csv"
9,Tabel 10,Akurasi terhadap empat pembanding,"exp05c, groupB_inference","exp05c_..._comparison.csv, _dm.csv, groupB_pharma_*.csv"



exp10 sampai exp14 TIDAK dipakai paper IJIES: ketiganya dijalankan setelah
naskah ronde 2 dikirim, dan hanya masuk disertasi.


## 7. Status ramalan pra-registrasi

Ramalan yang **gagal** ditulis apa adanya. Bunyi setiap ramalan disalin dari sumber
aslinya: P1–P3 dari sel markdown pertama `notebooks/exp05c_pharma_arlrx.ipynb` pada commit
`0b18ac2`, dan P4–P9 dari `PREREG_exp10_exp13_exp14.md`.

In [9]:
# Status seluruh ramalan pra-registrasi, disalin apa adanya dari
# RESULTS_exp10_exp13_exp14.md (P4-P9) dan
# notebooks/exp05c_pharma_arlrx.ipynb commit 0b18ac2 (P1-P3).
RAMALAN = [
 ("P1", "exp05c", "Gerbang akan menutup (w* = 0) pada sebagian besar kategori",
  "7 dari 8 kategori (minimal satu konfigurasi); 17 dari 32 konfigurasi", "Terbukti"),
 ("P2", "exp05c", "Augmentasi akan hampir tidak membantu", "Sesuai ramalan", "Terbukti"),
 ("P3", "exp05c", "gate_val_gain_pct akan tetap memprediksi perbaikan test",
  "r = -0,904 (arah berlawanan)", "GAGAL"),
 ("P4a", "exp13", "RMSE uji LSTM grid lebar berubah paling banyak +/-2% terhadap exp12", "-0,15%", "Terbukti"),
 ("P4b", "exp13", "AR-LRX tidak lebih buruk dari LSTM grid lebar lebih dari 1%", "-0,65% harian; -1,38% mingguan", "Terbukti"),
 ("P5a", "exp14", "AR-LRX tidak lebih buruk dari ETS bergulir lebih dari 1%", "+4,91% harian; +0,69% mingguan", "GAGAL (harian)"),
 ("P5b", "exp14", "ETS tidak lebih akurat daripada ARIMA bergulir pada data harian", "ETS -4,27%", "GAGAL"),
 ("P6a", "exp10", "Bagian gerbang tertutup pada alpha=0,50 >= bagian pada run bersih", "0,48 vs 0,53", "GAGAL"),
 ("P6b", "exp10", "Perubahan rata-rata bergerbang terhadap S1 <= +0,5% dan di bawah hybrid tanpa gerbang", "0,28 / 0,04 / 0,14% vs 2,55 / 1,07 / 1,83%", "Terbukti"),
 ("P7a", "exp10", "resid_val_r2 Rossmann menurun monoton terhadap noise", "0,238 -> 0,206 -> 0,159 -> 0,068", "Terbukti"),
 ("P7b", "exp10", "Keunggulan atas S1 [struktural] menyusut monoton", "-23,03 -> -20,30 -> -11,98 -> -4,39%", "Terbukti"),
 ("P7c", "exp10", "Tidak pernah lebih buruk dari S1 lebih dari 0,5%", "selalu lebih baik", "Terbukti"),
 ("P7d", "exp10", "Pada alpha <= 0,25 tetap unggul atas XGBoost dan LightGBM", "-6,08 / -5,42% dan -5,45 / -4,65%", "Terbukti"),
 ("P8a", "exp10", "Guncangan PharmaSales, baris terdampak: AR-LRX unggul atas XGBoost", "-4,44% (spike); -0,89% (level)", "Terbukti"),
 ("P8b", "exp10", "Guncangan PharmaSales, baris terdampak: AR-LRX vs S1 <= +2%", "+0,49%; +0,32%", "Terbukti"),
 ("P9a", "exp10", "Guncangan Rossmann, baris terdampak: AR-LRX-Aug unggul atas S1", "-29,90%; -33,33%", "Terbukti"),
 ("P9b", "exp10", "Guncangan Rossmann, baris terdampak: AR-LRX-Aug tidak kalah >2% dari XGBoost", "+3,43%; +8,15%", "GAGAL"),
]
r = pd.DataFrame(RAMALAN, columns=["Ramalan", "Eksperimen", "Pernyataan (ringkas)", "Teramati", "Hasil"])
display(r)
n_gagal = int(r.Hasil.str.startswith("GAGAL").sum())
print()
print(f"{len(r)} ramalan dicatat: {len(r) - n_gagal} terbukti, {n_gagal} GAGAL.")
print("Untuk P4-P9: sepuluh dari empat belas terkonfirmasi (RESULTS_exp10_exp13_exp14.md).")
print("Ramalan yang gagal TIDAK dihapus dan TIDAK ditulis ulang.")
print()
print("Bunyi ramalan disalin dari sumber aslinya: P1-P3 dari sel markdown pertama")
print("notebooks/exp05c_pharma_arlrx.ipynb pada commit 0b18ac2, dan P4-P9 dari")
print("PREREG_exp10_exp13_exp14.md.")

,Ramalan,Eksperimen,Pernyataan (ringkas),Teramati,Hasil
0,P1,exp05c,Gerbang akan menutup (w* = 0) pada sebagian besar kategori,7 dari 8 kategori (minimal satu konfigurasi); 17 dari 32 konfigurasi,Terbukti
1,P2,exp05c,Augmentasi akan hampir tidak membantu,Sesuai ramalan,Terbukti
2,P3,exp05c,gate_val_gain_pct akan tetap memprediksi perbaikan test,"r = -0,904 (arah berlawanan)",GAGAL
3,P4a,exp13,RMSE uji LSTM grid lebar berubah paling banyak +/-2% terhadap exp12,"-0,15%",Terbukti
4,P4b,exp13,AR-LRX tidak lebih buruk dari LSTM grid lebar lebih dari 1%,"-0,65% harian; -1,38% mingguan",Terbukti
5,P5a,exp14,AR-LRX tidak lebih buruk dari ETS bergulir lebih dari 1%,"+4,91% harian; +0,69% mingguan",GAGAL (harian)
6,P5b,exp14,ETS tidak lebih akurat daripada ARIMA bergulir pada data harian,"ETS -4,27%",GAGAL
7,P6a,exp10,"Bagian gerbang tertutup pada alpha=0,50 >= bagian pada run bersih","0,48 vs 0,53",GAGAL
8,P6b,exp10,"Perubahan rata-rata bergerbang terhadap S1 <= +0,5% dan di bawah hybrid tanpa gerbang","0,28 / 0,04 / 0,14% vs 2,55 / 1,07 / 1,83%",Terbukti
9,P7a,exp10,resid_val_r2 Rossmann menurun monoton terhadap noise,"0,238 -> 0,206 -> 0,159 -> 0,068",Terbukti



17 ramalan dicatat: 12 terbukti, 5 GAGAL.
Untuk P4-P9: sepuluh dari empat belas terkonfirmasi (RESULTS_exp10_exp13_exp14.md).
Ramalan yang gagal TIDAK dihapus dan TIDAK ditulis ulang.

Bunyi ramalan disalin dari sumber aslinya: P1-P3 dari sel markdown pertama
notebooks/exp05c_pharma_arlrx.ipynb pada commit 0b18ac2, dan P4-P9 dari
PREREG_exp10_exp13_exp14.md.


### 7.1 Jejak waktu P1–P3

P4–P9 punya berkas pra-registrasi tersendiri dengan commit terpisah sebelum hasil.
P1–P3 tidak. Bunyinya berada di sel markdown pertama `notebooks/exp05c_pharma_arlrx.ipynb`,
sehingga jejaknya perlu ditelusuri lewat riwayat commit.

In [10]:
# Jejak waktu P1-P3. Berbeda dengan P4-P9, ramalan P1-P3 tidak punya berkas
# pra-registrasi terpisah; bunyinya ada di sel markdown pertama notebook exp05c.
JEJAK = [
 ("0b18ac2", "30 Agustus 2026", "new framework ar-lrx",
  "Bunyi P1-P3 masuk ke notebooks/exp05c_pharma_arlrx.ipynb. Notebook di-commit TANPA keluaran, "
  "sehingga hasilnya belum terlihat pada commit ini."),
 ("276bf13", "4 September 2026", "Replace stub exp05a results with the real Windows run; ...",
  "Hasil exp05c dan keluaran notebook masuk. Inilah commit pertama yang memuat hasil P1-P3."),
 ("6c5e8aa", "16 September 2026", "Re-run AR-LRX experiments with cross-fitted residuals (R1-1)",
  "Dijalankan ulang dengan residual cross-fitted. P1 berubah dari 15 menjadi 17 dari 32 konfigurasi; "
  "P3 dari r = -0,834 menjadi r = -0,904."),
]
j = pd.DataFrame(JEJAK, columns=["Commit", "Tanggal", "Judul commit", "Yang terjadi"])
for _, x in j.iterrows():
    print("=" * 100)
    print(f"{x.Commit}   {x.Tanggal}   {x['Judul commit']}")
    print(f"   {x['Yang terjadi']}")
print("=" * 100)
print()
print("Urutan bunyi-ramalan lebih dahulu, baru hasil, karena itu terdokumentasi untuk P1-P3:")
print("commit 0b18ac2 memuat ramalannya tanpa keluaran, dan hasilnya baru masuk di 276bf13.")
print()
print("CATATAN KRITERIA. Bunyi P1 adalah 'gerbang akan menutup pada sebagian besar")
print("kategori'. Kriteria otomatis di dalam notebook exp05c lebih longgar daripada")
print("bunyi itu: sel ringkasannya memakai `p1 = tutup > 0`, yaitu TERBUKTI bila w* = 0")
print("muncul pada sekurang-kurangnya satu konfigurasi. Angka yang dilaporkan di sini")
print("dihitung menurut bunyi aslinya, per kategori.")

0b18ac2   30 Agustus 2026   new framework ar-lrx
   Bunyi P1-P3 masuk ke notebooks/exp05c_pharma_arlrx.ipynb. Notebook di-commit TANPA keluaran, sehingga hasilnya belum terlihat pada commit ini.
276bf13   4 September 2026   Replace stub exp05a results with the real Windows run; ...
   Hasil exp05c dan keluaran notebook masuk. Inilah commit pertama yang memuat hasil P1-P3.
6c5e8aa   16 September 2026   Re-run AR-LRX experiments with cross-fitted residuals (R1-1)
   Dijalankan ulang dengan residual cross-fitted. P1 berubah dari 15 menjadi 17 dari 32 konfigurasi; P3 dari r = -0,834 menjadi r = -0,904.

Urutan bunyi-ramalan lebih dahulu, baru hasil, karena itu terdokumentasi untuk P1-P3:
commit 0b18ac2 memuat ramalannya tanpa keluaran, dan hasilnya baru masuk di 276bf13.

CATATAN KRITERIA. Bunyi P1 adalah 'gerbang akan menutup pada sebagian besar
kategori'. Kriteria otomatis di dalam notebook exp05c lebih longgar daripada
bunyi itu: sel ringkasannya memakai `p1 = tutup > 0`, yaitu TERBUKTI

Perhitungan P1 menurut bunyi aslinya, yaitu per kategori:

In [11]:
# Hitung ulang P1 menurut bunyi aslinya, per kategori.
c5 = baca("exp05c_crossfit_crossfit.csv")
ar = c5[c5.model == "AR-LRX [linear]"]
per = ar.groupby("category").gate_w.apply(lambda s: int((s == 0).sum()))
tab = per.rename("konfigurasi dengan w* = 0 (dari 4)").reset_index()
display(tab)
print()
print(f"Kategori dengan minimal satu konfigurasi tertutup : {int((per >= 1).sum())} dari {per.size}")
print(f"Kategori dengan seluruh konfigurasi tertutup      : {int((per == 4).sum())} dari {per.size}")
print(f"Konfigurasi tertutup, seluruhnya                  : {int((ar.gate_w == 0).sum())} dari {len(ar)}")

,category,konfigurasi dengan w* = 0 (dari 4)
0,M01AB,2
1,M01AE,1
2,N02BA,0
3,N02BE,4
4,N05B,2
5,N05C,2
6,R03,2
7,R06,4



Kategori dengan minimal satu konfigurasi tertutup : 7 dari 8
Kategori dengan seluruh konfigurasi tertutup      : 2 dari 8
Konfigurasi tertutup, seluruhnya                  : 17 dari 32


## 8. Catatan reproduksi

Hal-hal yang harus disampaikan sendiri, sebelum ditanyakan.

In [12]:
# Catatan reproduksi yang harus disampaikan apa adanya kepada promotor.
CATATAN = [
 ("exp01, exp02, exp03, exp04",
  "Menjalankan ulang notebooknya HARI INI tidak menghasilkan angka yang diarsipkan.",
  "Baris 'LR-XGB (residual)' memakai P.fp_lr_xgb_residual dengan nilai bawaan, yang sejak revisi R1-1 "
  "berarti residual cross-fitted; dan baris LightGBM memakai B.fp_lightgbm yang kini subsample_freq=1. "
  "Keduanya berubah setelah berkas hasil diarsipkan.",
  "Lewatkan cross_fit_folds=None dan subsample_freq=0 secara eksplisit (REPRODUCE.md bagian 8). "
  "Baris lain pada notebook itu tidak terpengaruh."),
 ("exp07 (Tabel 5.7 dan 5.8)",
  "Angka kedua tabel diketik langsung di sumber naskah, bukan dibaca dari CSV.",
  "content_bab5.py hanya memanggil read_csv untuk exp05c_crossfit_crossfit.csv, exp08_summary.csv dan "
  "kedua exp10_summary_*.csv. Tabel 5.7 dan 5.8 berisi daftar literal.",
  "Notebook pelapor exp07 memeriksa setiap angka literal itu terhadap exp07_summary_*.csv satu per satu."),
 ("exp13 dan seluruh model neural",
  "Tidak reproduksi bit-per-bit, bahkan pada mesin yang sama.",
  "Kernel CPU TensorFlow yang multithread tidak deterministik. Seed sudah dipasang "
  "(tf.random.set_seed dan keras.utils.set_random_seed pada setiap fit), tetapi "
  "tf.config.experimental.enable_op_determinism() tidak dipakai.",
  "Dinyatakan sebagai keterbatasan pada naskah. Angka yang dilaporkan berasal dari satu run pada mesin referensi."),
 ("Seluruh komponen berbasis pohon",
  "Reproduksi bit-per-bit pada satu sistem operasi, TIDAK lintas sistem operasi.",
  "Dengan versi pustaka yang sama, XGBoost dengan subsample/colsample < 1 memberi fit yang sedikit berbeda "
  "di Windows dan di Linux; pada exp05c bedanya di digit signifikan keempat, cukup untuk mengubah sebagian "
  "konfigurasi yang terpilih pada data validasi.",
  "Seluruh angka naskah berasal dari satu lingkungan: Windows, Python 3.10.11, CPU."),
 ("Berkas *_predictions.npz",
  "Tidak dilacak git.",
  ".gitignore mengabaikan *.npz karena ukurannya. groupB_inference.py dan sebagian gambar membacanya.",
  "Berkas itu harus berasal dari run asli; klon baru tidak memilikinya."),
 ("results/logs/*.log",
  "Tidak dilacak git.",
  ".gitignore mengabaikan *.log. Log inilah yang memuat runtime dan spesifikasi terpilih exp10, exp13, exp14.",
  "Notebook pelapor membacanya bila ada, dan berjalan normal bila tidak ada."),
 ("_revision_r2/ dan arlrx_pipeline/",
  "Sumber naskah tidak ikut diversikan di repositori ini.",
  "Keduanya diabaikan git. Naskah ronde 2 IJIES dan toolchain perakit naskah ada di luar repositori.",
  "Bila promotor meminta jejak naskah, sumbernya harus disediakan terpisah."),
]
c = pd.DataFrame(CATATAN, columns=["Menyangkut", "Apa yang harus disampaikan", "Sebabnya", "Yang harus dilakukan"])
for _, r in c.iterrows():
    print("=" * 100)
    print(f"{r['Menyangkut']}")
    print(f"  Apa    : {r['Apa yang harus disampaikan']}")
    print(f"  Sebab  : {r['Sebabnya']}")
    print(f"  Aksi   : {r['Yang harus dilakukan']}")

exp01, exp02, exp03, exp04
  Apa    : Menjalankan ulang notebooknya HARI INI tidak menghasilkan angka yang diarsipkan.
  Sebab  : Baris 'LR-XGB (residual)' memakai P.fp_lr_xgb_residual dengan nilai bawaan, yang sejak revisi R1-1 berarti residual cross-fitted; dan baris LightGBM memakai B.fp_lightgbm yang kini subsample_freq=1. Keduanya berubah setelah berkas hasil diarsipkan.
  Aksi   : Lewatkan cross_fit_folds=None dan subsample_freq=0 secara eksplisit (REPRODUCE.md bagian 8). Baris lain pada notebook itu tidak terpengaruh.
exp07 (Tabel 5.7 dan 5.8)
  Apa    : Angka kedua tabel diketik langsung di sumber naskah, bukan dibaca dari CSV.
  Sebab  : content_bab5.py hanya memanggil read_csv untuk exp05c_crossfit_crossfit.csv, exp08_summary.csv dan kedua exp10_summary_*.csv. Tabel 5.7 dan 5.8 berisi daftar literal.
  Aksi   : Notebook pelapor exp07 memeriksa setiap angka literal itu terhadap exp07_summary_*.csv satu per satu.
exp13 dan seluruh model neural
  Apa    : Tidak reproduksi bit-pe

## 9. Sidik jari integritas `results/`

In [13]:
# Sidik jari seluruh results/. Dipakai untuk menunjukkan bahwa notebook pelapor
# tidak mengubah satu berkas hasil pun.
baris = []
for f in sorted(RES.rglob("*")):
    if f.is_file():
        h = hashlib.sha256(f.read_bytes()).hexdigest().upper()
        baris.append({"berkas": f.relative_to(ROOT).as_posix(), "bytes": f.stat().st_size, "sha256": h})
sidik = pd.DataFrame(baris)
gab = hashlib.sha256("\n".join(f"{r.sha256}  {r.berkas}" for r in sidik.itertuples()).encode()).hexdigest().upper()

# Dicatat di luar repositori SEBELUM notebook pelapor mana pun dibuat.
SIDIK_AWAL = "1474B7A30155B39E9C60B250B7655A926BAC9AD40F4329ED2B113EEAB650501D"

print(f"Berkas di results/    : {len(sidik)}")
print(f"Total ukuran          : {sidik.bytes.sum():,} bytes")
print(f"Sidik jari SEBELUM    : {SIDIK_AWAL}")
print(f"Sidik jari SEKARANG   : {gab}")
print()
if gab == SIDIK_AWAL:
    print("IDENTIK. Tidak ada satu pun berkas hasil yang berubah sejak snapshot diambil.")
else:
    print("!" * 78)
    print("SIDIK JARI BERBEDA. Ada berkas di results/ yang berubah. Laporkan, jangan dibetulkan.")
    print("!" * 78)
print()
print("Sidik jari gabungan = SHA-256 atas daftar '<sha256>  <path relatif root>',")
print("diurutkan menurut path. Daftar lengkapnya disimpan di luar repositori.")
print()
display(sidik.head(12))

Berkas di results/    : 179
Total ukuran          : 155,058,890 bytes
Sidik jari SEBELUM    : 1474B7A30155B39E9C60B250B7655A926BAC9AD40F4329ED2B113EEAB650501D
Sidik jari SEKARANG   : 1474B7A30155B39E9C60B250B7655A926BAC9AD40F4329ED2B113EEAB650501D

IDENTIK. Tidak ada satu pun berkas hasil yang berubah sejak snapshot diambil.

Sidik jari gabungan = SHA-256 atas daftar '<sha256>  <path relatif root>',
diurutkan menurut path. Daftar lengkapnya disimpan di luar repositori.



,berkas,bytes,sha256
0,results/_dryrun_container/exp01_pharma_daily.csv,64577,521E28A875C963A440AEBA8DB41306D98D465F29EA452D1E896FE9567DEEAB43
1,results/_dryrun_container/exp01_pharma_daily.meta.json,507,C2F287FB6020EC56DC51C03BA6423BA91232A3F3B50BBC353C90061DBBB2EEB5
2,results/_dryrun_container/exp01_pharma_daily_dm_test.csv,1887,5B55A0BA947E295563851A50F54152BF109FB94207E90871E0C58A8F5017E4D6
3,results/_dryrun_container/exp01_pharma_daily_lag_ablation.csv,36848,22F6F31BD466750E29205303699B45AA0ADE99CFAECB8062537B094ADD9CCE5B
4,results/_dryrun_container/exp01_pharma_daily_lag_ablation.meta.json,519,45DF518188A20DFA9E0C5B2C8C6F76BD4D3AA06580D9E70B5366833588331F92
5,results/_dryrun_container/exp01_pharma_daily_splits.csv,1991,0626A35AD1D47FECA4E41EDE53AC35486903ACD97DDD90608D2D48646EBFE291
6,results/_dryrun_container/exp02_pharma_weekly.csv,64234,A4268F1649BB56A6F0A971AD935A5BAE80DF4B00DAE17936FFE03CCC222C2251
7,results/_dryrun_container/exp02_pharma_weekly.meta.json,508,7B6C3C52A8DD36CE5EC321C4BF6AE1BB575193C7CFFB5D0D7454262B5907170C
8,results/_dryrun_container/exp02_pharma_weekly_dm_test.csv,1962,957A78971DC61035B4E3E1DC0BD35357848FC3769576729961E2EB403ABEFE12
9,results/_dryrun_container/exp02_pharma_weekly_lag_ablation.csv,36579,1BF1F89B9DCB4AEEE7537D6084B565FB0A34BCC9253405D04D96491796218BE2


---

### Ringkasan untuk promotor

- Ada **17 eksperimen bernomor** (exp01–exp08, exp10–exp14; nomor exp09 tidak terpakai)
  ditambah **8 skrip audit dan alat**.
- Seluruh angka Bab IV dan Bab V dapat ditelusuri ke berkas di `results/`. Perkecualiannya
  adalah Tabel 5.7 dan 5.8, yang angkanya diketik langsung di sumber naskah, sehingga
  kecocokannya diperiksa terpisah di notebook pelapor exp07.
- Empat belas ramalan pra-registrasi P4–P9 dilaporkan, dengan **sepuluh terkonfirmasi dan
  empat gagal**. Ramalan yang gagal tetap dicantumkan.
- Tiga catatan reproduksi wajib disampaikan sendiri: notebook exp01–exp04 tidak lagi
  mereproduksi baris arsipnya bila dijalankan apa adanya, model neural tidak deterministik,
  dan hasil berbasis pohon hanya reproduksi bit-per-bit pada satu sistem operasi.